<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/llm/Create-Embeddings-out-of-an-LLM/embeddings-q2.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("create-embeddings-out-of-an-llm", ...)`


# Create Embeddings out of an LLM

### Problem Statement
Your mission, should you choose to accept it, is to extract **meaningful sentence-level embeddings** using a pre-trained **causal language model (SmolLM2-135M)** on Amazon Reviews.

You're working with a **generative language model**, but you’re not here to generate Shakespeare. Instead, you’ll tap into its **hidden states** to get semantic embeddings that capture the essence of a review — the good, the bad, and the brutally honest.

---

### Requirements

1. **Load and Tokenize Text**
   - Use the `McAuley-Lab/Amazon-Reviews-2023` dataset (All_Beauty reviews, `raw/review_categories/All_Beauty.jsonl`).
   - Load ~10 sample reviews for testing (stream them; the full file is ~330 MB).
   - Tokenize them using `"HuggingFaceTB/SmolLM2-135M"` tokenizer.

2. **Extract Embeddings**
   - Run the tokenized batch through the model with `output_hidden_states=True`.
   - Access the **last hidden layer** from `outputs.hidden_states[-1]`.

3. **Compute Sentence Embeddings** — implement two pooling functions. Each takes the final-layer states `hidden_states` `(batch, seq_len, hidden_dim)` and the tokenizer's `attention_mask` `(batch, seq_len)` (1 = real token, 0 = padding; any integer or bool dtype), returns one vector per text, `(batch, hidden_dim)`, and leaves `hidden_states` unmodified:
   - **`mean_pool(hidden_states, attention_mask)`**: average the final-layer token states, **excluding padding tokens** — divide each row by its number of real tokens, not by `seq_len`.
   - **`last_token_pool(hidden_states, attention_mask)`**: the state of each row's **last real token**. An encoder such as BERT has a `[CLS]` token whose state is trained to summarize the input. A causal LM has no such token, but because of the causal mask its last real token is the only one that has attended to the whole text, so it is the natural analog (LLM-based embedding models such as E5-Mistral pool it, after contrastive fine-tuning). Which index that is depends on the padding side: the function must work for **right and left padding** without being told which one was used.
   - Mean pooling over a causal model weights early tokens, which saw little context, as much as late ones; SGPT uses a position-weighted mean for that reason.

4. **Find the cosine similarity for a given keyword**
   - Embed the keyword the same way and compute its cosine similarity with each review embedding.
   - Raw hidden states of a generative LM that was never trained for embeddings are anisotropic: almost all cosine similarities come out high. Compare the *ranking* of the reviews, not the absolute values.

---

### Grading

`check("create-embeddings-out-of-an-llm", mean_pool=mean_pool, last_token_pool=last_token_pool)` downloads nothing. It tests the two functions on random hidden states whose padded positions hold huge junk values, with right- and left-padded batches, and through a tiny randomly initialised causal LM: a text's embedding must be the same whether it is embedded alone or batched with longer and shorter texts.

---

### Constraints

- ❌ Do **not** use sentence-transformers or pre-built embedding tools like `bert-as-service`.
- ❌ Do **not** generate text (no `.generate()`).
- ✅ Use only Hugging Face's `AutoModelForCausalLM` and `AutoTokenizer`.
- ✅ Exclude padding tokens when computing average embeddings.
- ✅ Ensure everything runs on `cuda` if available.

---

<details>
  <summary>💡 Hint</summary>

  - Run the model with `output_hidden_states=True`; `outputs.hidden_states[-1]` is the last layer, `(batch_size, seq_len, hidden_dim)`.
  - The attention mask says which positions are real. Broadcast it to `(batch, seq_len, 1)` to zero out the padded states, sum over the sequence, and divide by the number of real tokens per row.
  - The last real token is at the last position where the mask is 1 — with right padding that is the number of real tokens minus one, with left padding it is always `seq_len - 1`.

</details>

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
def mean_pool(hidden_states, attention_mask):
    """
    Args:
        hidden_states:  (batch, seq_len, hidden_dim) final-layer token states.
        attention_mask: (batch, seq_len), 1 for real tokens and 0 for padding (int or bool).

    Returns:
        (batch, hidden_dim): the mean of each row's real-token states (padding excluded).
    """
    # TODO
    ...


def last_token_pool(hidden_states, attention_mask):
    """
    Args:
        hidden_states:  (batch, seq_len, hidden_dim) final-layer token states.
        attention_mask: (batch, seq_len), 1 for real tokens and 0 for padding (int or bool).

    Returns:
        (batch, hidden_dim): the state of each row's last real token, for right or left padding.
    """
    # TODO
    ...

In [ ]:
# Self-check on a hand-made batch (no downloads): the same three texts, padded right and left
torch.manual_seed(0)
lengths = [3, 1, 4]
right = torch.tensor([[1, 1, 1, 0], [1, 0, 0, 0], [1, 1, 1, 1]])
left = torch.tensor([[0, 1, 1, 1], [0, 0, 0, 1], [1, 1, 1, 1]])
h_right = torch.randn(3, 4, 8)
h_right[right == 0] = 1e4                    # padded positions hold junk, which must not leak
h_left = torch.full_like(h_right, 1e4)
for i, n in enumerate(lengths):
    h_left[i, 4 - n:] = h_right[i, :n]       # the same real tokens, moved to the right end

expected_mean = torch.stack([h_right[i, :n].mean(dim=0) for i, n in enumerate(lengths)])
expected_last = torch.stack([h_right[i, n - 1] for i, n in enumerate(lengths)])
for h, mask, side in ((h_right, right, "right"), (h_left, left, "left")):
    assert mean_pool(h, mask).shape == (3, 8), "one (hidden_dim,) vector per text"
    assert torch.allclose(mean_pool(h, mask), expected_mean, atol=1e-4), f"mean_pool, {side} padding"
    assert torch.allclose(last_token_pool(h, mask), expected_last), f"last_token_pool, {side} padding"
print("Self-check passed")

In [ ]:
# Load ~10 reviews from Amazon-Reviews-2023 (All_Beauty). Streaming the raw JSONL file
# downloads only the first few rows of the ~330 MB file and needs no dataset script.
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/All_Beauty.jsonl",
    split="train",
    streaming=True,
)
reviews = [row["text"] for row in dataset.take(10)]  # list of 10 review strings

In [ ]:
for i, review in enumerate(reviews[:3]):
    print(f"#{i + 1}: {review[:120]}")

In [ ]:
# Load SmolLM2-135M model and tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")

# SmolLM2's tokenizer has no padding token; batching needs one. Reusing EOS is safe
# because padded positions are masked out by the attention mask (and pooling below).
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

In [ ]:
def embed(texts):
    """Return (mean-pooled, last-token) sentence embeddings, each (batch, hidden_dim)."""
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**enc, output_hidden_states=True)
    last_hidden = outputs.hidden_states[-1].float()          # (batch, seq_len, hidden_dim)
    mask = enc["attention_mask"]                              # 1 = real token, 0 = padding
    return mean_pool(last_hidden, mask), last_token_pool(last_hidden, mask)


sentence_embeddings, last_token_embeddings = embed(reviews)
print("Sentence embeddings shape:", sentence_embeddings.shape)  # (10, hidden_dim)

# --- Cosine similarity for a given keyword ---
keyword = "quality"
keyword_embedding, _ = embed([keyword])                          # (1, hidden_dim)

cosine_similarities = F.cosine_similarity(sentence_embeddings, keyword_embedding)  # (10,)

# Raw LLM hidden states are anisotropic: every cosine similarity tends to be high,
# so compare the ranking, not the absolute values.
for i in cosine_similarities.argsort(descending=True).tolist():
    print(f"{cosine_similarities[i].item():.4f}  {reviews[i][:100]}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("create-embeddings-out-of-an-llm", mean_pool=mean_pool, last_token_pool=last_token_pool)